In [ ]:
# ============================================================
# OLLAMA SETUP
# ============================================================
#
# Purpose:
#   1. Install Ollama if necessary
#   2. Start Ollama server
#   3. Check GPU
#   4. Download evaluator model
#   5. Download embedding model
#   6. Test both models
#
# ============================================================

import os
import subprocess
import time
import requests


# ============================================================
# CONFIGURATION
# ============================================================

OLLAMA_HOST = "127.0.0.1:11434"

OLLAMA_API = f"http://{OLLAMA_HOST}"

EVALUATOR_MODEL = "qwen2.5:14b"

EMBEDDING_MODEL = "nomic-embed-text"


print("=" * 80)
print("CELL 1 — OLLAMA SETUP")
print("=" * 80)


# ============================================================
# 1. CHECK GPU
# ============================================================

print("\n" + "=" * 80)
print("1. CHECKING NVIDIA GPU")
print("=" * 80)

gpu_check = subprocess.run(
    ["bash", "-c", "command -v nvidia-smi"],
    capture_output=True,
    text=True
)

if gpu_check.returncode == 0:

    gpu_result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.used",
            "--format=csv,noheader"
        ],
        capture_output=True,
        text=True
    )

    if gpu_result.returncode == 0:

        GPU_INFO = gpu_result.stdout.strip()

        print("✓ GPU detected")
        print(GPU_INFO)

    else:

        GPU_INFO = "NVIDIA GPU detected"

        print("⚠ nvidia-smi failed")

else:

    GPU_INFO = "GPU NOT DETECTED"

    print("⚠ NVIDIA GPU not detected")


# ============================================================
# 2. INSTALL OLLAMA
# ============================================================

print("\n" + "=" * 80)
print("2. CHECKING OLLAMA")
print("=" * 80)

ollama_check = subprocess.run(
    ["bash", "-c", "command -v ollama"],
    capture_output=True,
    text=True
)

if ollama_check.returncode == 0:

    print("✓ Ollama already installed")

else:

    print("Installing Ollama...")

    subprocess.run(
        [
            "bash",
            "-c",
            "curl -fsSL https://ollama.com/install.sh | sh"
        ],
        check=True
    )

    print("✓ Ollama installed")


# ============================================================
# 3. START OLLAMA SERVER
# ============================================================

print("\n" + "=" * 80)
print("3. STARTING OLLAMA SERVER")
print("=" * 80)

os.environ["OLLAMA_HOST"] = OLLAMA_HOST

ollama_running = False

try:

    response = requests.get(
        f"{OLLAMA_API}/api/tags",
        timeout=3
    )

    if response.status_code == 200:
        ollama_running = True

except Exception:
    pass


if ollama_running:

    print("✓ Ollama server already running")

else:

    print("Starting Ollama server...")

    ollama_log = open(
        "/tmp/ollama.log",
        "w"
    )

    subprocess.Popen(
        ["ollama", "serve"],
        stdout=ollama_log,
        stderr=subprocess.STDOUT
    )

    # Wait for server
    for second in range(60):

        try:

            response = requests.get(
                f"{OLLAMA_API}/api/tags",
                timeout=2
            )

            if response.status_code == 200:

                ollama_running = True

                print(
                    f"✓ Ollama server started "
                    f"after {second + 1} seconds"
                )

                break

        except Exception:
            time.sleep(1)

    if not ollama_running:

        print("\nOllama log:")

        if os.path.exists("/tmp/ollama.log"):

            with open(
                "/tmp/ollama.log",
                "r"
            ) as f:

                print(f.read()[-5000:])

        raise RuntimeError(
            "Ollama server could not be started."
        )


# ============================================================
# 4. CHECK CURRENT MODELS
# ============================================================

print("\n" + "=" * 80)
print("4. CHECKING OLLAMA MODELS")
print("=" * 80)

response = requests.get(
    f"{OLLAMA_API}/api/tags",
    timeout=10
)

response.raise_for_status()

models_data = response.json()

installed_models = [
    model.get("name", "")
    for model in models_data.get("models", [])
]

if installed_models:

    print("Installed models:")

    for model in installed_models:
        print("  •", model)

else:

    print("No models currently installed.")


# ============================================================
# 5. PULL EVALUATOR MODEL
# ============================================================

print("\n" + "=" * 80)
print(f"5. CHECKING {EVALUATOR_MODEL}")
print("=" * 80)

if not any(
    name == EVALUATOR_MODEL
    or name.startswith(EVALUATOR_MODEL + ":")
    for name in installed_models
):

    print(f"Downloading {EVALUATOR_MODEL}...")

    subprocess.run(
        ["ollama", "pull", EVALUATOR_MODEL],
        check=True
    )

    print(f"✓ {EVALUATOR_MODEL} downloaded")

else:

    print(f"✓ {EVALUATOR_MODEL} already installed")


# ============================================================
# 6. PULL EMBEDDING MODEL
# ============================================================

print("\n" + "=" * 80)
print(f"6. CHECKING {EMBEDDING_MODEL}")
print("=" * 80)

if not any(
    name == EMBEDDING_MODEL
    or name.startswith(EMBEDDING_MODEL + ":")
    for name in installed_models
):

    print(f"Downloading {EMBEDDING_MODEL}...")

    subprocess.run(
        ["ollama", "pull", EMBEDDING_MODEL],
        check=True
    )

    print(f"✓ {EMBEDDING_MODEL} downloaded")

else:

    print(f"✓ {EMBEDDING_MODEL} already installed")


# ============================================================
# 7. FINAL MODEL LIST
# ============================================================

print("\n" + "=" * 80)
print("7. FINAL OLLAMA MODELS")
print("=" * 80)

subprocess.run(
    ["ollama", "list"],
    check=True
)


# ============================================================
# 8. TEST EVALUATOR MODEL
# ============================================================

print("\n" + "=" * 80)
print("8. TESTING EVALUATOR MODEL")
print("=" * 80)

test_result = subprocess.run(
    [
        "ollama",
        "run",
        EVALUATOR_MODEL,
        "Reply with exactly: OLLAMA_TEST_OK"
    ],
    capture_output=True,
    text=True
)

if test_result.returncode != 0:

    print(test_result.stderr)

    raise RuntimeError(
        "Evaluator model test failed."
    )

print("Model response:")
print(test_result.stdout.strip())

print("✓ Evaluator model works")


# ============================================================
# 9. TEST EMBEDDING MODEL
# ============================================================

print("\n" + "=" * 80)
print("9. TESTING EMBEDDING MODEL")
print("=" * 80)

embedding_response = requests.post(
    f"{OLLAMA_API}/api/embeddings",
    json={
        "model": EMBEDDING_MODEL,
        "prompt": "Embedding test"
    },
    timeout=120
)

embedding_response.raise_for_status()

embedding_data = embedding_response.json()

embedding_vector = embedding_data.get(
    "embedding"
)

if not embedding_vector:

    raise RuntimeError(
        "Embedding model returned no embedding."
    )

print(
    "Embedding dimension:",
    len(embedding_vector)
)

print("✓ Embedding model works")


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 80)
print("CELL 1 COMPLETE")
print("=" * 80)

print("✓ Ollama installed")
print("✓ Ollama server running")
print("✓ Evaluator model:", EVALUATOR_MODEL)
print("✓ Embedding model:", EMBEDDING_MODEL)
print("✓ GPU:", GPU_INFO)

print("\nProceed to CELL 2.")

CELL 1 — OLLAMA SETUP

1. CHECKING NVIDIA GPU
✓ GPU detected
Tesla T4, 15360 MiB, 0 MiB

2. CHECKING OLLAMA
✓ Ollama already installed

3. STARTING OLLAMA SERVER
Starting Ollama server...
✓ Ollama server started after 4 seconds

4. CHECKING OLLAMA MODELS
Installed models:
  • nomic-embed-text:latest
  • qwen2.5:14b
  • llama3.1:8b

5. CHECKING qwen2.5:14b
✓ qwen2.5:14b already installed

6. CHECKING nomic-embed-text
✓ nomic-embed-text already installed

7. FINAL OLLAMA MODELS

8. TESTING EVALUATOR MODEL
Model response:
OLLAMA_TEST_OK
✓ Evaluator model works

9. TESTING EMBEDDING MODEL
Embedding dimension: 768
✓ Embedding model works

CELL 1 COMPLETE
✓ Ollama installed
✓ Ollama server running
✓ Evaluator model: qwen2.5:14b
✓ Embedding model: nomic-embed-text
✓ GPU: Tesla T4, 15360 MiB, 0 MiB

Proceed to CELL 2.


In [ ]:
import subprocess
import time
import requests

OLLAMA_API = "http://127.0.0.1:11434"

print("Starting Ollama server...")

# Check whether Ollama is already running
try:
    r = requests.get(f"{OLLAMA_API}/api/tags", timeout=5)
    print("✓ Ollama is already running")
except:
    print("Ollama is not running. Starting it...")

    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    # Wait for Ollama
    for i in range(30):
        time.sleep(1)
        try:
            r = requests.get(f"{OLLAMA_API}/api/tags", timeout=2)
            if r.status_code == 200:
                print(f"✓ Ollama started after {i+1} seconds")
                break
        except:
            pass
    else:
        raise RuntimeError("❌ Ollama could not be started")

# Show installed models
data = requests.get(f"{OLLAMA_API}/api/tags").json()

print("\nInstalled Ollama models:")
for model in data.get("models", []):
    print("  ✓", model["name"])

Starting Ollama server...
✓ Ollama is already running

Installed Ollama models:
  ✓ nomic-embed-text:latest
  ✓ qwen2.5:14b
  ✓ llama3.1:8b


In [ ]:
# ============================================================
# COMPLETE RAGAS SETUP
# ============================================================
# Installs:
#   - Ollama
#   - Qwen2.5:14b
#   - nomic-embed-text
#   - RAGAS 0.4.3
#   - OpenAI 1.109.1
#
# Creates:
#   - evaluator_llm
#   - evaluator_embeddings
#   - faithfulness
#   - answer_correctness
#   - context_precision
#   - context_recall
#
# After this cell finishes successfully:
# RUN YOUR RAGAS EVALUATION CELL
# ============================================================

import os
import sys
import subprocess
import time
import types
import requests

print("=" * 80)
print("COMPLETE RAGAS + OLLAMA SETUP")
print("=" * 80)


# ============================================================
# 1. CONFIGURATION
# ============================================================

OLLAMA_HOST = "127.0.0.1"
OLLAMA_PORT = 11434
OLLAMA_API = f"http://{OLLAMA_HOST}:{OLLAMA_PORT}"

OPENAI_BASE_URL = f"{OLLAMA_API}/v1"

EVALUATOR_MODEL = "qwen2.5:14b"
EMBEDDING_MODEL = "nomic-embed-text"

print("\nConfiguration:")
print("  Ollama       :", OLLAMA_API)
print("  Evaluator    :", EVALUATOR_MODEL)
print("  Embeddings   :", EMBEDDING_MODEL)


# ============================================================
# 2. CHECK GPU
# ============================================================

print("\n" + "=" * 80)
print("GPU")
print("=" * 80)

try:

    gpu = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total",
            "--format=csv,noheader"
        ],
        capture_output=True,
        text=True
    )

    if gpu.returncode == 0:
        print("✓ GPU detected")
        print(gpu.stdout.strip())
    else:
        print("⚠ NVIDIA GPU not detected")

except Exception:
    print("⚠ nvidia-smi unavailable")


# ============================================================
# 3. INSTALL OLLAMA
# ============================================================

print("\n" + "=" * 80)
print("OLLAMA")
print("=" * 80)

check_ollama = subprocess.run(
    ["bash", "-lc", "command -v ollama"],
    capture_output=True,
    text=True
)

if check_ollama.returncode == 0:

    print("✓ Ollama already installed")

else:

    print("Installing Ollama...")

    result = subprocess.run(
        [
            "bash",
            "-lc",
            "curl -fsSL https://ollama.com/install.sh | sh"
        ],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError("Ollama installation failed")

    print("✓ Ollama installed")


# ============================================================
# 4. START OLLAMA
# ============================================================

print("\nStarting Ollama server...")

ollama_running = False

try:

    response = requests.get(
        f"{OLLAMA_API}/api/tags",
        timeout=3
    )

    if response.status_code == 200:
        ollama_running = True
        print("✓ Ollama is already running")

except Exception:
    pass


if not ollama_running:

    env = os.environ.copy()
    env["OLLAMA_HOST"] = f"{OLLAMA_HOST}:{OLLAMA_PORT}"

    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env=env
    )

    for second in range(1, 61):

        time.sleep(1)

        try:

            response = requests.get(
                f"{OLLAMA_API}/api/tags",
                timeout=2
            )

            if response.status_code == 200:

                ollama_running = True

                print(
                    f"✓ Ollama started after "
                    f"{second} seconds"
                )

                break

        except Exception:
            pass


if not ollama_running:

    raise RuntimeError(
        "❌ Ollama server could not be started"
    )


# ============================================================
# 5. CHECK / DOWNLOAD MODELS
# ============================================================

print("\n" + "=" * 80)
print("OLLAMA MODELS")
print("=" * 80)


def get_models():

    response = requests.get(
        f"{OLLAMA_API}/api/tags",
        timeout=10
    )

    response.raise_for_status()

    return [
        model["name"]
        for model in response.json().get("models", [])
    ]


installed_models = get_models()

print("Installed models:")

if installed_models:

    for model in installed_models:
        print("  ✓", model)

else:

    print("  None")


required_models = [
    EVALUATOR_MODEL,
    EMBEDDING_MODEL
]


for model in required_models:

    exists = (
        model in installed_models
        or f"{model}:latest" in installed_models
    )

    if exists:

        print(f"\n✓ {model} already installed")

    else:

        print(
            f"\n⬇ Downloading {model}..."
        )

        result = subprocess.run(
            ["ollama", "pull", model]
        )

        if result.returncode != 0:

            raise RuntimeError(
                f"Failed to download {model}"
            )

        print(
            f"✓ {model} downloaded"
        )


print("\nFinal models:")

subprocess.run(
    ["ollama", "list"]
)


# ============================================================
# 6. INSTALL PYTHON PACKAGES
# ============================================================

print("\n" + "=" * 80)
print("PYTHON PACKAGES")
print("=" * 80)

packages = [
    "ragas==0.4.3",
    "openai==1.109.1",
    "langchain-openai",
    "langchain-ollama",
    "nest-asyncio",
    "instructor"
]

print("Installing required packages...")

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *packages
    ],
    capture_output=True,
    text=True
)

if result.returncode != 0:

    print(result.stdout)
    print(result.stderr)

    raise RuntimeError(
        "Python package installation failed"
    )

print("✓ Packages installed")


# ============================================================
# 7. RAGAS 0.4.3 COMPATIBILITY PATCH
# ============================================================

print("\n" + "=" * 80)
print("RAGAS COMPATIBILITY PATCH")
print("=" * 80)

# RAGAS 0.4.3 tries to import an old VertexAI module.
# We don't use VertexAI, so create a temporary compatibility module.

vertex_module = types.ModuleType(
    "langchain_community.chat_models.vertexai"
)


class DummyChatVertexAI:
    pass


vertex_module.ChatVertexAI = DummyChatVertexAI

sys.modules[
    "langchain_community.chat_models.vertexai"
] = vertex_module


try:

    import langchain_community.llms as community_llms

    if not hasattr(
        community_llms,
        "VertexAI"
    ):

        class DummyVertexAI:
            pass

        community_llms.VertexAI = DummyVertexAI

    print("✓ Compatibility patch applied")

except Exception as e:

    print(
        "⚠ Compatibility patch warning:",
        e
    )


# ============================================================
# 8. IMPORT RAGAS
# ============================================================

print("\n" + "=" * 80)
print("RAGAS")
print("=" * 80)

import nest_asyncio

nest_asyncio.apply()

import ragas

print(
    "✓ RAGAS version:",
    ragas.__version__
)


# ============================================================
# 9. IMPORT FACTORIES
# ============================================================

from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory

print("✓ RAGAS factories imported")


# ============================================================
# 10. OPENAI CLIENT
# ============================================================

from openai import OpenAI, AsyncOpenAI

sync_client = OpenAI(
    base_url=OPENAI_BASE_URL,
    api_key="ollama"
)

async_client = AsyncOpenAI(
    base_url=OPENAI_BASE_URL,
    api_key="ollama"
)

print("✓ OpenAI clients created")


# ============================================================
# 11. CREATE RAGAS LLM
# ============================================================

print("\nCreating evaluator LLM...")

evaluator_llm = llm_factory(
    EVALUATOR_MODEL,
    provider="openai",
    client=async_client,
    max_tokens=4096,
    temperature=0
)

print("✓ evaluator_llm created")


# ============================================================
# 12. CREATE EMBEDDINGS
# ============================================================

print("\nCreating embedding model...")

evaluator_embeddings = embedding_factory(
    "openai",
    model=EMBEDDING_MODEL,
    client=async_client,
    interface="modern"
)

print("✓ evaluator_embeddings created")


# ============================================================
# 13. CREATE ALL FOUR METRICS
# ============================================================

print("\n" + "=" * 80)
print("CREATING RAGAS METRICS")
print("=" * 80)

from ragas.metrics.collections import (
    Faithfulness,
    AnswerCorrectness,
    ContextPrecision,
    ContextRecall
)


# IMPORTANT:
# These are the exact variables used by Cell 2.

faithfulness = Faithfulness(
    llm=evaluator_llm
)

print("✓ faithfulness")


answer_correctness = AnswerCorrectness(
    llm=evaluator_llm,
    embeddings=evaluator_embeddings
)

print("✓ answer_correctness")


context_precision = ContextPrecision(
    llm=evaluator_llm
)

print("✓ context_precision")


context_recall = ContextRecall(
    llm=evaluator_llm
)

print("✓ context_recall")


# Also keep them in one dictionary
metrics = {
    "faithfulness": faithfulness,
    "answer_correctness": answer_correctness,
    "context_precision": context_precision,
    "context_recall": context_recall
}


# ============================================================
# 14. VERIFY ALL VARIABLES
# ============================================================

print("\n" + "=" * 80)
print("VERIFYING SETUP")
print("=" * 80)

required_objects = [
    "evaluator_llm",
    "evaluator_embeddings",
    "faithfulness",
    "answer_correctness",
    "context_precision",
    "context_recall"
]

for name in required_objects:

    if name not in globals():

        raise RuntimeError(
            f"❌ {name} was not created"
        )

    print(
        f"✓ {name}"
    )


# ============================================================
# 15. TEST QWEN
# ============================================================

print("\n" + "=" * 80)
print("TESTING QWEN")
print("=" * 80)

try:

    response = sync_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "user",
                "content": "Reply with exactly: OK"
            }
        ],
        temperature=0,
        max_tokens=5
    )

    print(
        "Qwen response:",
        response.choices[0].message.content
    )

    print("✓ Qwen API works")

except Exception as e:

    raise RuntimeError(
        f"❌ Qwen API test failed: {e}"
    )


# ============================================================
# 16. TEST EMBEDDINGS
# ============================================================

print("\n" + "=" * 80)
print("TESTING EMBEDDINGS")
print("=" * 80)

try:

    response = sync_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input="This is a test sentence."
    )

    vector = response.data[0].embedding

    print(
        "✓ Embedding API works"
    )

    print(
        "Vector dimensions:",
        len(vector)
    )

except Exception as e:

    raise RuntimeError(
        f"❌ Embedding API test failed: {e}"
    )


# ============================================================
# 17. FINAL STATUS
# ============================================================

print("\n")
print("=" * 80)
print("✓✓✓ SETUP COMPLETE ✓✓✓")
print("=" * 80)

print("\nOllama:")
print("  ✓ Server running")

print("\nModels:")
print(f"  ✓ {EVALUATOR_MODEL}")
print(f"  ✓ {EMBEDDING_MODEL}")

print("\nRAGAS:")
print(f"  ✓ Version {ragas.__version__}")

print("\nRAGAS objects:")
print("  ✓ evaluator_llm")
print("  ✓ evaluator_embeddings")
print("  ✓ faithfulness")
print("  ✓ answer_correctness")
print("  ✓ context_precision")
print("  ✓ context_recall")

print("\n" + "=" * 80)
print("READY FOR YOUR EVALUATION CELL")
print("=" * 80)

COMPLETE RAGAS + OLLAMA SETUP

Configuration:
  Ollama       : http://127.0.0.1:11434
  Evaluator    : qwen2.5:14b
  Embeddings   : nomic-embed-text

GPU
✓ GPU detected
Tesla T4, 15360 MiB

OLLAMA
✓ Ollama already installed

Starting Ollama server...
✓ Ollama is already running

OLLAMA MODELS
Installed models:
  ✓ nomic-embed-text:latest
  ✓ qwen2.5:14b
  ✓ llama3.1:8b

✓ qwen2.5:14b already installed

✓ nomic-embed-text already installed

Final models:

PYTHON PACKAGES
Installing required packages...
✓ Packages installed

RAGAS COMPATIBILITY PATCH


/tmp/ipykernel_32483/1668841647.py:348: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community.llms as community_llms


✓ Compatibility patch applied

RAGAS


/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✓ RAGAS version: 0.4.3
✓ RAGAS factories imported
✓ OpenAI clients created

Creating evaluator LLM...
✓ evaluator_llm created

Creating embedding model...


/tmp/ipykernel_32483/1668841647.py:442: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = embedding_factory(


✓ evaluator_embeddings created

CREATING RAGAS METRICS
✓ faithfulness
✓ answer_correctness
✓ context_precision
✓ context_recall

VERIFYING SETUP
✓ evaluator_llm
✓ evaluator_embeddings
✓ faithfulness
✓ answer_correctness
✓ context_precision
✓ context_recall

TESTING QWEN
Qwen response: OK
✓ Qwen API works

TESTING EMBEDDINGS
✓ Embedding API works
Vector dimensions: 768


✓✓✓ SETUP COMPLETE ✓✓✓

Ollama:
  ✓ Server running

Models:
  ✓ qwen2.5:14b
  ✓ nomic-embed-text

RAGAS:
  ✓ Version 0.4.3

RAGAS objects:
  ✓ evaluator_llm
  ✓ evaluator_embeddings
  ✓ faithfulness
  ✓ answer_correctness
  ✓ context_precision
  ✓ context_recall

READY FOR YOUR EVALUATION CELL


In [ ]:
# ============================================================
#  RAGAS EVALUATION
# ============================================================
# RUN AFTER THE COMPLETE SETUP CELL
#
# The setup cell must already have created:
#
#   evaluator_llm
#   evaluator_embeddings
#   faithfulness
#   answer_correctness
#   context_precision
#   context_recall
#
# This cell:
#   1. Loads the CSV
#   2. Selects 80 questions
#   3. Evaluates 4 RAGAS metrics
#   4. Saves after every question
#   5. Supports resume
#   6. Saves errors separately
#   7. Calculates final averages
# ============================================================

import os
import ast
import json
import time
from datetime import datetime

import pandas as pd
import numpy as np


# ============================================================
# 1. CONFIGURATION
# ============================================================

BASE_DIR = "/content/drive/MyDrive"

PREDICTIONS_FILE = os.path.join(
    BASE_DIR,
    "ragas_predictions.csv"
)

RESULTS_FILE = os.path.join(
    BASE_DIR,
    "ragas_results_28-9.csv"
)

OVERALL_FILE = os.path.join(
    BASE_DIR,
    "ragas_overall_scores_28-9.csv"
)

ERROR_FILE = os.path.join(
    BASE_DIR,
    "ragas_evaluation_errors_28-9.csv"
)

NUMBER_OF_QUESTIONS = 80

EVALUATOR_MODEL = "qwen2.5:14b"

EMBEDDING_MODEL = "nomic-embed-text"

METRIC_NAMES = [
    "faithfulness",
    "answer_correctness",
    "context_precision",
    "context_recall"
]


# ============================================================
# 2. VERIFY SETUP CELL
# ============================================================

print("=" * 80)
print("CHECKING RAGAS SETUP")
print("=" * 80)

required_objects = [
    "evaluator_llm",
    "evaluator_embeddings",
    "faithfulness",
    "answer_correctness",
    "context_precision",
    "context_recall"
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:

    raise RuntimeError(
        "\nMissing objects:\n"
        + "\n".join(
            f"  ❌ {x}"
            for x in missing_objects
        )
        + "\n\nRun the COMPLETE SETUP CELL first."
    )

print("✓ evaluator_llm")
print("✓ evaluator_embeddings")
print("✓ faithfulness")
print("✓ answer_correctness")
print("✓ context_precision")
print("✓ context_recall")

print("\n✓ RAGAS setup is ready")


# ============================================================
# 3. CHECK INPUT CSV
# ============================================================

print("\n" + "=" * 80)
print("LOADING INPUT DATA")
print("=" * 80)

if not os.path.exists(PREDICTIONS_FILE):

    raise FileNotFoundError(
        f"\nInput file not found:\n{PREDICTIONS_FILE}"
    )

df = pd.read_csv(
    PREDICTIONS_FILE
)

print("✓ Input file found")
print("Total rows:", len(df))


# ============================================================
# 4. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "question",
    "answer",
    "ground_truth",
    "contexts"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:

    raise ValueError(
        "Missing required columns:\n"
        + "\n".join(
            f"  - {x}"
            for x in missing_columns
        )
    )

print("✓ Required columns found")


# ============================================================
# 5. SELECT 80 QUESTIONS
# ============================================================

df = df.iloc[
    :NUMBER_OF_QUESTIONS
].copy()

print(
    f"✓ Questions selected: {len(df)}"
)


# ============================================================
# 6. CLEAN TEXT COLUMNS
# ============================================================

df["question"] = (
    df["question"]
    .fillna("")
    .astype(str)
)

df["answer"] = (
    df["answer"]
    .fillna("")
    .astype(str)
)

df["ground_truth"] = (
    df["ground_truth"]
    .fillna("")
    .astype(str)
)


# ============================================================
# 7. PARSE CONTEXTS
# ============================================================

def parse_contexts(value):

    if value is None:
        return []

    try:

        if pd.isna(value):
            return []

    except Exception:
        pass


    if isinstance(value, list):

        return [
            str(x)
            for x in value
            if str(x).strip()
        ]


    text = str(value).strip()


    # Python list
    try:

        parsed = ast.literal_eval(text)

        if isinstance(parsed, list):

            return [
                str(x)
                for x in parsed
                if str(x).strip()
            ]

    except Exception:
        pass


    # JSON list
    try:

        parsed = json.loads(text)

        if isinstance(parsed, list):

            return [
                str(x)
                for x in parsed
                if str(x).strip()
            ]

    except Exception:
        pass


    # Single context
    if text:

        return [text]


    return []


df["contexts"] = (
    df["contexts"]
    .apply(parse_contexts)
)

print("✓ Contexts parsed")


# ============================================================
# 8. CREATE METRIC DICTIONARY
# ============================================================

metrics = {

    "faithfulness":
        faithfulness,

    "answer_correctness":
        answer_correctness,

    "context_precision":
        context_precision,

    "context_recall":
        context_recall
}


# ============================================================
# 9. EVALUATE ONE METRIC
# ============================================================

async def evaluate_metric(
    metric_name,
    question,
    answer,
    contexts,
    ground_truth
):

    metric = metrics[
        metric_name
    ]


    # --------------------------------------------------------
    # Faithfulness
    # --------------------------------------------------------

    if metric_name == "faithfulness":

        result = await metric.ascore(

            user_input=question,

            response=answer,

            retrieved_contexts=contexts
        )


    # --------------------------------------------------------
    # Answer Correctness
    # --------------------------------------------------------

    elif metric_name == "answer_correctness":

        result = await metric.ascore(

            user_input=question,

            response=answer,

            reference=ground_truth
        )


    # --------------------------------------------------------
    # Context Precision
    # --------------------------------------------------------

    elif metric_name == "context_precision":

        result = await metric.ascore(

            user_input=question,

            retrieved_contexts=contexts,

            reference=ground_truth
        )


    # --------------------------------------------------------
    # Context Recall
    # --------------------------------------------------------

    elif metric_name == "context_recall":

        result = await metric.ascore(

            user_input=question,

            retrieved_contexts=contexts,

            reference=ground_truth
        )


    else:

        raise ValueError(
            f"Unknown metric: {metric_name}"
        )


    # RAGAS score objects contain .value
    return float(result.value)


# ============================================================
# 10. LOAD PREVIOUS RESULTS
# ============================================================

print("\n" + "=" * 80)
print("CHECKING PREVIOUS RESULTS")
print("=" * 80)

if os.path.exists(RESULTS_FILE):

    try:

        results_df = pd.read_csv(
            RESULTS_FILE
        )

        print(
            "Previous results found:",
            len(results_df)
        )

    except Exception as e:

        print(
            "⚠ Could not read previous results:"
        )

        print(e)

        results_df = pd.DataFrame()

else:

    results_df = pd.DataFrame()

    print(
        "No previous results found."
    )


# ============================================================
# 11. LOAD ERROR LOG
# ============================================================

if os.path.exists(ERROR_FILE):

    try:

        error_df = pd.read_csv(
            ERROR_FILE
        )

        errors = error_df.to_dict(
            "records"
        )

        print(
            f"Previous errors: {len(errors)}"
        )

    except Exception:

        errors = []

else:

    errors = []


# ============================================================
# 12. CHECK WHETHER QUESTION IS COMPLETE
# ============================================================

def question_already_completed(
    question
):

    if results_df.empty:

        return False


    if "question" not in results_df.columns:

        return False


    matches = (
        results_df["question"]
        .astype(str)
        .str.strip()
        .eq(
            str(question).strip()
        )
    )


    if not matches.any():

        return False


    existing = (
        results_df.loc[matches]
        .iloc[-1]
    )


    for metric_name in METRIC_NAMES:

        if metric_name not in existing.index:

            return False


        value = existing[
            metric_name
        ]


        if pd.isna(value):

            return False


    return True


# ============================================================
# 13. TEST QUESTION 1
# ============================================================

print("\n" + "=" * 80)
print("TESTING QUESTION 1")
print("=" * 80)

test_row = df.iloc[0]

print(
    "\nQuestion:"
)

print(
    test_row["question"]
)

print(
    "\nNumber of contexts:",
    len(test_row["contexts"])
)


test_scores = {}


for metric_name in METRIC_NAMES:

    print("\n" + "-" * 70)

    print(
        "Testing:",
        metric_name
    )

    start = time.time()


    try:

        score = await evaluate_metric(

            metric_name,

            test_row["question"],

            test_row["answer"],

            test_row["contexts"],

            test_row["ground_truth"]
        )


        elapsed = time.time() - start

        test_scores[
            metric_name
        ] = score


        print(
            f"✓ {metric_name}: "
            f"{score:.4f}"
        )

        print(
            f"Time: "
            f"{elapsed / 60:.2f} minutes"
        )


    except Exception as e:

        print(
            f"\n❌ {metric_name} FAILED"
        )

        print(
            type(e).__name__,
            ":",
            str(e)
        )

        raise


print("\n" + "=" * 80)
print("✓ QUESTION 1 TEST SUCCESSFUL")
print("=" * 80)

for metric_name in METRIC_NAMES:

    print(
        f"{metric_name:25s}: "
        f"{test_scores[metric_name]:.4f}"
    )


# ============================================================
# 14. FULL EVALUATION
# ============================================================

print("\n" + "=" * 80)
print("STARTING FULL EVALUATION")
print("=" * 80)

print(
    f"Questions : {len(df)}"
)

print(
    "Metrics   : 4"
)

print(
    "Evaluator :",
    EVALUATOR_MODEL
)

print(
    "Embedding :",
    EMBEDDING_MODEL
)


for index in range(
    len(df)
):

    question_number = index + 1

    row = df.iloc[index]

    question = row["question"]

    answer = row["answer"]

    contexts = row["contexts"]

    ground_truth = row["ground_truth"]


    # --------------------------------------------------------
    # RESUME
    # --------------------------------------------------------

    if question_already_completed(
        question
    ):

        print(
            f"\n[{question_number}/{len(df)}] "
            "✓ Already completed — SKIPPING"
        )

        continue


    # --------------------------------------------------------
    # QUESTION HEADER
    # --------------------------------------------------------

    print("\n" + "=" * 80)

    print(
        f"QUESTION "
        f"{question_number}/"
        f"{len(df)}"
    )

    print("=" * 80)

    print(
        question[:500]
    )


    # --------------------------------------------------------
    # RESULT ROW
    # --------------------------------------------------------

    row_result = {

        "question_number":
            question_number,

        "question":
            question,

        "answer":
            answer,

        "ground_truth":
            ground_truth,

        "faithfulness":
            np.nan,

        "answer_correctness":
            np.nan,

        "context_precision":
            np.nan,

        "context_recall":
            np.nan,

        "evaluator_model":
            EVALUATOR_MODEL,

        "embedding_model":
            EMBEDDING_MODEL,

        "timestamp":
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            )
    }


    # Keep paper_id if available
    if "paper_id" in df.columns:

        row_result[
            "paper_id"
        ] = row["paper_id"]


    # --------------------------------------------------------
    # RUN EACH METRIC
    # --------------------------------------------------------

    for metric_name in METRIC_NAMES:

        print(
            "\n" + "-" * 70
        )

        print(
            "Running:",
            metric_name
        )

        start = time.time()


        try:

            score = await evaluate_metric(

                metric_name,

                question,

                answer,

                contexts,

                ground_truth
            )


            elapsed = time.time() - start


            row_result[
                metric_name
            ] = score


            print(
                f"✓ {metric_name}"
            )

            print(
                f"Score: {score:.4f}"
            )

            print(
                f"Time: "
                f"{elapsed / 60:.2f} minutes"
            )


        except Exception as e:

            elapsed = time.time() - start


            print(
                f"❌ {metric_name} FAILED"
            )

            print(
                "Error:",
                type(e).__name__,
                str(e)
            )


            errors.append({

                "question_number":
                    question_number,

                "question":
                    question,

                "metric":
                    metric_name,

                "error_type":
                    type(e).__name__,

                "error":
                    str(e),

                "timestamp":
                    datetime.now().strftime(
                        "%Y-%m-%d %H:%M:%S"
                    )
            })


            # Save error immediately
            pd.DataFrame(
                errors
            ).to_csv(
                ERROR_FILE,
                index=False
            )


    # --------------------------------------------------------
    # UPDATE RESULTS
    # --------------------------------------------------------

    new_result = pd.DataFrame(
        [row_result]
    )


    if results_df.empty:

        results_df = new_result

    else:

        # Remove previous incomplete row
        if "question" in results_df.columns:

            results_df = results_df[
                results_df["question"]
                .astype(str)
                .str.strip()
                .ne(
                    str(question).strip()
                )
            ]


        results_df = pd.concat(
            [
                results_df,
                new_result
            ],
            ignore_index=True
        )


    # --------------------------------------------------------
    # SAVE IMMEDIATELY
    # --------------------------------------------------------

    results_df.to_csv(
        RESULTS_FILE,
        index=False
    )


    print("\n" + "=" * 80)

    print(
        f"✓ QUESTION "
        f"{question_number} SAVED"
    )

    print("=" * 80)


    for metric_name in METRIC_NAMES:

        value = row_result[
            metric_name
        ]


        if pd.isna(value):

            print(
                f"{metric_name:25s}: ERROR"
            )

        else:

            print(
                f"{metric_name:25s}: "
                f"{value:.4f}"
            )


# ============================================================
# 15. FINAL RESULTS
# ============================================================

print("\n" + "=" * 80)
print("CALCULATING FINAL RESULTS")
print("=" * 80)


if not os.path.exists(
    RESULTS_FILE
):

    raise RuntimeError(
        "No results file was created."
    )


final_results = pd.read_csv(
    RESULTS_FILE
)


# ============================================================
# 16. OVERALL SCORES
# ============================================================

overall = {

    "evaluator_model":
        EVALUATOR_MODEL,

    "embedding_model":
        EMBEDDING_MODEL,

    "questions_in_results":
        len(final_results)
}


for metric_name in METRIC_NAMES:

    if metric_name not in final_results.columns:

        overall[
            metric_name
        ] = None

        overall[
            metric_name
            + "_valid_questions"
        ] = 0

        continue


    values = pd.to_numeric(

        final_results[
            metric_name
        ],

        errors="coerce"
    )


    valid = values.dropna()


    if len(valid) > 0:

        overall[
            metric_name
        ] = float(
            valid.mean()
        )

    else:

        overall[
            metric_name
        ] = None


    overall[
        metric_name
        + "_valid_questions"
    ] = len(valid)


# ============================================================
# 17. SAVE OVERALL
# ============================================================

overall_df = pd.DataFrame(
    [overall]
)

overall_df.to_csv(
    OVERALL_FILE,
    index=False
)


# ============================================================
# 18. FINAL DISPLAY
# ============================================================

print("\n" + "=" * 80)
print("FINAL RAGAS RESULTS")
print("=" * 80)

print()

for metric_name in METRIC_NAMES:

    value = overall[
        metric_name
    ]

    valid = overall[
        metric_name
        + "_valid_questions"
    ]


    if value is not None:

        print(
            f"{metric_name:25s}: "
            f"{value:.4f} "
            f"(valid: {valid})"
        )

    else:

        print(
            f"{metric_name:25s}: "
            "NO RESULT"
        )


# ============================================================
# 19. FILES
# ============================================================

print("\n" + "=" * 80)
print("OUTPUT FILES")
print("=" * 80)

print(
    "\nResults:"
)

print(
    RESULTS_FILE
)

print(
    "\nOverall:"
)

print(
    OVERALL_FILE
)

print(
    "\nErrors:"
)

print(
    ERROR_FILE
)


# ============================================================
# 20. ERROR SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("ERROR SUMMARY")
print("=" * 80)

if errors:

    print(
        f"⚠ Total errors: {len(errors)}"
    )

    print(
        f"See:\n{ERROR_FILE}"
    )

else:

    print(
        "✓ No evaluation errors"
    )


print("\n" + "=" * 80)
print("✓ EVALUATION COMPLETE")
print("=" * 80)

CHECKING RAGAS SETUP
✓ evaluator_llm
✓ evaluator_embeddings
✓ faithfulness
✓ answer_correctness
✓ context_precision
✓ context_recall

✓ RAGAS setup is ready

LOADING INPUT DATA
✓ Input file found
Total rows: 80
✓ Required columns found
✓ Questions selected: 80
✓ Contexts parsed

CHECKING PREVIOUS RESULTS
Previous results found: 80
Previous errors: 400

TESTING QUESTION 1

Question:
Does the study use a simplified model potential?

Number of contexts: 20

----------------------------------------------------------------------
Testing: faithfulness
✓ faithfulness: 1.0000
Time: 0.84 minutes

----------------------------------------------------------------------
Testing: answer_correctness
✓ answer_correctness: 0.3687
Time: 0.93 minutes

----------------------------------------------------------------------
Testing: context_precision
✓ context_precision: 0.3571
Time: 2.21 minutes

----------------------------------------------------------------------
Testing: context_recall
✓ context_recall

CancelledError: 